In [ ]:
!pip install tensorflow

In [ ]:
# Install packages if needed (uncomment when running in a fresh environment)
# !pip install -q kagglehub tensorflow scikit-learn seaborn matplotlib pandas

import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices("GPU"))

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhavikjikadara/dog-and-cat-classification-dataset")
print("Path to dataset files:", path)

In [ ]:
# Find the directory containing the Cat and Dog folders.
# The dataset is copied into a clean RGB-only version before TensorFlow reads it.

from PIL import Image

def find_class_directory(root):
    root = Path(root)
    candidates = [root] + [p for p in root.rglob("*") if p.is_dir()]

    for folder in candidates:
        try:
            subdirs = [p for p in folder.iterdir() if p.is_dir()]
        except PermissionError:
            continue

        names = {p.name.lower() for p in subdirs}
        if {"cat", "dog"}.issubset(names):
            return folder

    for folder in candidates:
        try:
            subdirs = [p for p in folder.iterdir() if p.is_dir()]
        except PermissionError:
            continue

        names = {p.name.lower() for p in subdirs}
        if {"cats", "dogs"}.issubset(names):
            return folder

    raise FileNotFoundError(
        "Could not find the directory containing the Cat/Dog class folders."
    )

data_dir = find_class_directory(path)
print("Original image directory:", data_dir)
print("Class folders:", [p.name for p in data_dir.iterdir() if p.is_dir()])


# Convert all readable images to RGB JPEGs.
# This fixes dataset images with unusual channel counts (for example 2-channel images).
clean_data_dir = Path("/tmp/cats_dogs_clean_rgb")
if clean_data_dir.exists():
    import shutil
    shutil.rmtree(clean_data_dir)

clean_data_dir.mkdir(parents=True, exist_ok=True)

valid_extensions = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".webp"}

converted = 0
skipped = 0

for class_dir in data_dir.iterdir():
    if not class_dir.is_dir():
        continue

    output_class_dir = clean_data_dir / class_dir.name
    output_class_dir.mkdir(parents=True, exist_ok=True)

    for img_path in class_dir.rglob("*"):
        if img_path.suffix.lower() not in valid_extensions:
            continue

        try:
            with Image.open(img_path) as img:
                img = img.convert("RGB")

                # Give each output image a unique filename to avoid collisions.
                relative_name = "_".join(img_path.relative_to(class_dir).parts)
                output_path = output_class_dir / f"{Path(relative_name).stem}.jpg"

                img.save(output_path, format="JPEG", quality=95)
                converted += 1

        except Exception as e:
            skipped += 1

print(f"Clean RGB images created: {converted}")
print(f"Skipped unreadable/invalid images: {skipped}")
print("Clean dataset directory:", clean_data_dir)

In [ ]:
IMG_SIZE = (150, 150)
BATCH_SIZE = 32
SEED = 42

# Use the cleaned RGB dataset.
train_ds, val_ds = tf.keras.utils.image_dataset_from_directory(
    clean_data_dir,
    labels="inferred",
    label_mode="binary",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.20,
    subset="both",
    seed=SEED,
    shuffle=True
)

class_names = train_ds.class_names
print("Classes:", class_names)

# Recreate validation data without shuffling so predictions/confusion matrix
# always correspond to the correct labels.
_, val_ds = tf.keras.utils.image_dataset_from_directory(
    clean_data_dir,
    labels="inferred",
    label_mode="binary",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    validation_split=0.20,
    subset="both",
    seed=SEED,
    shuffle=False
)

AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)


In [ ]:
# Preview a few training images
plt.figure(figsize=(10, 6))

for images, labels in train_ds.take(1):
    for i in range(min(8, len(images))):
        ax = plt.subplot(2, 4, i + 1)
        plt.imshow(tf.cast(images[i], tf.uint8))
        plt.title(class_names[int(labels[i].numpy()[0])])
        plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# VGG16-specific preprocessing.
# preprocess_input converts RGB -> BGR and subtracts ImageNet channel means.

def vgg_preprocess(images, labels):
    images = preprocess_input(tf.cast(images, tf.float32))
    return images, labels

# Scratch CNN uses simple 0-1 scaling.
def scratch_preprocess(images, labels):
    images = tf.cast(images, tf.float32) / 255.0
    return images, labels

vgg_train_ds = train_ds.map(
    vgg_preprocess,
    num_parallel_calls=AUTOTUNE
)
vgg_val_ds = val_ds.map(
    vgg_preprocess,
    num_parallel_calls=AUTOTUNE
)

scratch_train_ds = train_ds.map(
    scratch_preprocess,
    num_parallel_calls=AUTOTUNE
)
scratch_val_ds = val_ds.map(
    scratch_preprocess,
    num_parallel_calls=AUTOTUNE
)

vgg_train_ds = vgg_train_ds.prefetch(AUTOTUNE)
vgg_val_ds = vgg_val_ds.prefetch(AUTOTUNE)
scratch_train_ds = scratch_train_ds.prefetch(AUTOTUNE)
scratch_val_ds = scratch_val_ds.prefetch(AUTOTUNE)

In [ ]:
scratch_model = keras.Sequential([
    layers.Input(shape=(150, 150, 3)),

    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(1, activation="sigmoid")
], name="Scratch_CNN")

scratch_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

scratch_model.summary()

scratch_history = scratch_model.fit(
    scratch_train_ds,
    validation_data=scratch_val_ds,
    epochs=5
)


In [ ]:
vgg_base = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(150, 150, 3)
)

# Freeze the entire convolutional feature extractor.
vgg_base.trainable = False

vgg_model = keras.Sequential([
    vgg_base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(1, activation="sigmoid")
], name="VGG16_Transfer_Learning")

vgg_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

vgg_model.summary()

total_params = vgg_model.count_params()
trainable_params = np.sum([np.prod(v.shape) for v in vgg_model.trainable_weights])

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

In [ ]:
vgg_history = vgg_model.fit(
    vgg_train_ds,
    validation_data=vgg_val_ds,
    epochs=8
)